# 02 — Seleção e balanceamento das classes

Este notebook define quais classes têm suporte suficiente para participar do experimento sem consultar validação ou teste.

Para cada classe $i$, contamos somente suas janelas de treino, $c_i$, e trabalhamos com $z_i=\log(1+c_i)$. A transformação reduz a assimetria da distribuição: uma classe com milhares de exemplos deixa de dominar numericamente outra com dezenas. Calculamos então a média $\mu_z$ e o desvio-padrão $\sigma_z$ dessas contagens transformadas e definimos o corte

$$T_z = \mu_z-k\sigma_z, \qquad c_{\min}=\lceil e^{T_z}-1\rceil.$$

Aqui, **$k$ é o número de desvios-padrão que deslocamos o corte para baixo da média**. Ele não é o número de classes, não é aprendido pelo classificador e não tem relação com k-NN ou k-means. Com $k=0$, o corte fica na média das contagens em escala logarítmica. Com $k=1$, aceitamos classes até um desvio-padrão abaixo dela. Aumentar $k$ reduz $c_{\min}$ e inclui classes mais raras; diminuir $k$ torna o recorte mais conservador. Independentemente de $k$, uma classe ainda precisa aparecer no número mínimo de grupos de gravação, evitando selecionar uma espécie sustentada por muitas janelas de uma única origem.

O notebook mostra a análise de sensibilidade para vários valores de $k$, aplica o valor escolhido usando apenas o treino e congela o recorte. Depois, o treino é subamostrado de forma determinística para uma cota comum; validação e teste mantêm sua distribuição natural. A média e o desvio-padrão são usados como uma regra operacional interpretável na escala $\log(1+c)$ — não como afirmação de que as contagens originais seguem perfeitamente uma Gaussiana.

> Execute as células de cima para baixo. Parâmetros editáveis aparecem em uma célula própria e todo artefato gerado informa seu destino.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "data" / "freeze-spec-v1.json").is_file():
    ROOT = ROOT.parent
if not (ROOT / "data" / "freeze-spec-v1.json").is_file():
    raise RuntimeError("Execute este notebook dentro do repositório CodEco.")

for location in (ROOT / "src", ROOT / "scripts"):
    if str(location) not in sys.path:
        sys.path.insert(0, str(location))

REPORTS = ROOT / "reports"
DERIVED = ROOT / "data" / "derived"
REPORTS.mkdir(exist_ok=True)
DERIVED.mkdir(exist_ok=True)
print(f"Repositório: {ROOT}")


## Parâmetros editáveis

- `K_SIGMA` controla quão abaixo da média logarítmica colocamos o corte. **Quanto maior, mais inclusivo**: entram classes menos frequentes e a cota comum do treino diminui.
- `MINIMUM_GROUPS` é uma proteção independente contra pseudorreplicação: uma classe precisa ocorrer em pelo menos esse número de grupos de gravação.
- `SENSITIVITY_K` contém apenas os valores comparados na análise de sensibilidade; ele não executa uma busca orientada pelo teste.
- `SEED` torna a subamostragem do treino reproduzível.

O valor final de `K_SIGMA` deve ser decidido observando suporte e diversidade no **treino**, antes de examinar o desempenho no teste. Validação e teste não são balanceados artificialmente.


In [ ]:
K_SIGMA = {"ecosoundset": 1.0, "pteroset_ppa1": 1.0}
MINIMUM_GROUPS = {"ecosoundset": 3, "pteroset_ppa1": 3}
SEED = 42
SENSITIVITY_K = [0.0, 0.5, 1.0, 1.5, 2.0]
K_SIGMA, MINIMUM_GROUPS, SEED


## Sensibilidade do recorte

Esta tabela deve ser discutida antes de congelar `k_sigma`. Nenhum resultado de teste é usado para escolhê-lo.


In [ ]:
import pandas as pd
from codeco.corpora import (
    CORPUS_MANIFESTS, SelectionRule, deterministic_balanced_training_sample,
    freeze_selection, load_frozen_manifest, select_classes,
)

manifests = {name: load_frozen_manifest(name, ROOT) for name in CORPUS_MANIFESTS}
sensitivity = []
for corpus, df in manifests.items():
    for k in SENSITIVITY_K:
        _, meta = select_classes(df, SelectionRule(k_sigma=k, minimum_groups=MINIMUM_GROUPS[corpus], seed=SEED))
        sensitivity.append({"corpus": corpus, "k_sigma": k, "classes": meta["selected_class_count"], "cutoff": meta["count_cutoff"]})
sensitivity = pd.DataFrame(sensitivity)
sensitivity


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for corpus, group in sensitivity.groupby("corpus"):
    axes[0].plot(group.k_sigma, group.classes, "o-", label=corpus)
    axes[1].plot(group.k_sigma, group.cutoff, "o-", label=corpus)
axes[0].set(xlabel="k σ", ylabel="classes selecionadas", title="Número de classes")
axes[1].set(xlabel="k σ", ylabel="janelas mínimas no treino", title="Limiar de suporte")
axes[1].set_yscale("log")
for ax in axes: ax.legend()
fig.tight_layout()
plt.show()


## Recorte escolhido por corpus

A linha tracejada é o limiar derivado de `μ − kσ` em `log1p(contagem)`. A tabela exibe explicitamente cada classe incluída ou excluída.


In [ ]:
selection_outputs = {}
selection_tables = {}
for corpus, df in manifests.items():
    rule = SelectionRule(k_sigma=K_SIGMA[corpus], minimum_groups=MINIMUM_GROUPS[corpus], seed=SEED)
    table, meta = select_classes(df, rule)
    selected = meta["selected_classes"]
    quota = meta["count_cutoff"]
    sampled, balance = deterministic_balanced_training_sample(df, selected, quota=quota, seed=SEED)
    paths = freeze_selection(corpus, table, meta, sampled, balance, DERIVED / "selections")
    selection_outputs[corpus] = {key: str(path.relative_to(ROOT)) for key, path in paths.items()}
    selection_tables[corpus] = table
pd.DataFrame([
    {"corpus": corpus, **paths} for corpus, paths in selection_outputs.items()
])


In [ ]:
fig, axes = plt.subplots(len(selection_tables), 1, figsize=(12, 5 * len(selection_tables)))
if len(selection_tables) == 1: axes = [axes]
for ax, (corpus, table) in zip(axes, selection_tables.items()):
    colors = table.selected.map({True: "#2b8c5a", False: "#b8b8b8"})
    ax.bar(range(len(table)), table.windows, color=colors)
    ax.axhline(table.count_cutoff.iloc[0], color="#a33", linestyle="--", label="limiar")
    ax.set_yscale("log")
    ax.set_title(f"{corpus}: {int(table.selected.sum())} classes selecionadas")
    ax.set_ylabel("janelas de treino (log)")
    ax.legend()
fig.tight_layout()
fig.savefig(REPORTS / "selected_class_distributions.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
for corpus, table in selection_tables.items():
    print(f"\n{corpus}")
    display(table[["label", "windows", "groups", "z_score", "count_cutoff", "selected", "selection_reason"]])
